In [0]:
catalogo = "medalhao"
bronze_db_name = "bronze"
silver_db_name = "silver"
gold_db_name = "gold"

from pyspark.sql import functions as F
from pyspark.sql.types import *

def ingest_csv(nome_arquivo, nome_tabela):
    try:
        table_name = nome_tabela 
        landing_path = f"/Volumes/medalhao/landing/raw_data/{nome_arquivo}" #caminho da base de dados
        df = spark.read.csv(landing_path, header=True, inferSchema=True) #lendo a base de dados, inferSchema=True faz com que o spark entenda o tipo de dados e header que tem cabeçalho
        if df.count() == 0:
            raise ValueError (f"O arquivo {nome_arquivo} está Vazio.") #verificando se o arquivo tem algum conteúdo
        df_metadata = df.withColumn("timestamp_ingestion", F.current_timestamp()) #adicionando uma coluna com a data de ingestão dos dados
        df_metadata.write.format("delta").mode("overwrite").saveAsTable(f"{catalogo}.{table_name}") #escrevendo a base de dados no formato delta, overwrite faz com que ele substitua o conteúdo da tabela caso ela já exista e saveAsTable faz com que ele salve a base de dados no formato delta no caminho especificado
    except Exception as e:
        print(f"Ocorreu um erro ao processar o arquivo {nome_arquivo}: {str(e)}")

ingest_csv("olist_customers_dataset.csv", "bronze.tb_customers")
ingest_csv("olist_geolocation_dataset.csv", "bronze.tb_geolocation")
ingest_csv("olist_order_items_dataset.csv", "bronze.tb_order_items")
ingest_csv("olist_order_payments_dataset.csv", "bronze.tb_order_payments")
ingest_csv("olist_order_reviews_dataset.csv", "bronze.tb_order_reviews")
ingest_csv("olist_orders_dataset.csv", "bronze.tb_orders")
ingest_csv("olist_products_dataset.csv", "bronze.tb_products")
ingest_csv("olist_sellers_dataset.csv", "bronze.tb_sellers")
ingest_csv("product_category_name_translation.csv", "bronze.tb_product_category_name_translation")
